In [ ]:
from azure.core.credentials import AzureKeyCredential
from azure.search.documents import SearchClient
from openai import AzureOpenAI
import os
import pandas as pd
from datetime import datetime
import re
import numpy as np

# use full functions
def get_text_embeddings(text):
    client = AzureOpenAI(api_key=os.getenv('openai_api_key'),
                     api_version=os.getenv('openai_api_version'),
                     azure_endpoint=os.getenv('openai_azure_endpoint'))
    if pd.isna(text):
        print('text is None')
        response = None
    else: 
        response = client.embeddings.create(model = "text-embedding-ada-002", input = text). data[0].embedding

    return response

# Azure Cognitive Search credentials
search_service_endpoint = os.getenv('azure_search_service_new_endpoint')
search_api_key = os.getenv('azure_search_new_api_key')
index_name = os.getenv('azure_index_Whole')

search_client = SearchClient(endpoint=search_service_endpoint, index_name=index_name, credential=AzureKeyCredential(search_api_key))

# read the data. 
df_original =pd.read_csv(r'C:\Users\Naresh.Sampara\PycharmProjects\P13_Webpilot\20250430_Whole_text_data_semantic_chunk.csv')
df = df_original.replace(np.nan, 'None')
count = 0
for i in range(0,len(df),100):

    df_slice = df.loc[i:i+100,:]

    # Prepare the data to upload
    actions = []
    for index, row in df_slice.iterrows():
        
        chunk_embedding = get_text_embeddings(row['text'])
        global_embedding = get_text_embeddings(row['summary'])
        current_date = datetime.now().strftime("%Y%m%d")

        FileName = os.path.splitext(os.path.basename(row["doc_name"]))[0]

        ID = f"{FileName}--{current_date}-{count}"
        ID = re.sub(r'[^\w\-]', '_', ID)
        count+=1
        # "id", "content", "Type", "Framework", "Filename", "Framework_link", "Classification"
        action ={
            "id": ID,
            "Chunk_id": row["fileTitle"], 
            "chunk": row['text'],
            "doc_name": FileName, 
            "doc_link": row['doc_link'],
            "Framework": row["Framework"], 
            "framework_Name": row["framework_Name"],
            "classification": row["classification"],
            "embeddings": chunk_embedding,
        }

        actions.append(action)

        # upload the documents 
    results = search_client.upload_documents(documents = actions)
    
    print(f"Upload succeeded: {results[0].succeeded}, {i}")


